In [6]:
import os
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

DATA_DIR = "data"
OUTPUT_DIR = "output"

MODEL_PATH = os.path.join(DATA_DIR, "AIDOCell")
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set environmental variables so models are downloaded to the the model path rather than ~/.cache/huggingface
os.environ['HF_HOME'] = MODEL_PATH                        # Primary control
os.environ['HUGGINGFACE_HUB_CACHE'] = MODEL_PATH          # Hub downloads
os.environ['TRANSFORMERS_CACHE'] = MODEL_PATH             # Transformers-specific

from modelgenerator.backbones import aido_cell_3m, aido_cell_10m, aido_cell_100m

# local .py files
from AIDOCell import (
    load_aidocell,
    extract_model_weights,
    AIDOCELL_DEFS,
    
)
from utils import (
    compute_attention_from_weights,
    save_results,
    load_results,
    RESULTS_DEFS,
)


In [2]:
model_class = aido_cell_3m

In [3]:
model_name = model_class.__name__
print(f"\n{'='*60}")
print(f"Extracting: {model_name}")
print(f"{'='*60}")

# 1. Load model and data
print("\n1. Loading model and data...")
model, gene_annotations, model_metadata = load_aidocell(model_class)
print(f"   {len(gene_annotations)} genes, {model_metadata['n_layers']} layers")

# 2. Extract weights
print("2. Extracting weights...")
weights_dict = extract_model_weights(model)
print(f"   Embeddings: {weights_dict['gene_embedding'].shape}")
print(f"   Attention weights: {model_metadata['n_layers']} layers × 4 matrices (Q,K,V,O)")

# 3. Save results
print("3. Saving results...")
save_results(weights_dict, gene_annotations, model_metadata, OUTPUT_DIR, AIDOCELL_DEFS.MODEL_NAME)
print("   Successfully saved all results!")


INFO:AIDOCell:Loading AIDOCell model
/Users/sean/Desktop/GITHUB/napistu/lib/napistu-scrapyard/applications/foundation_models/.aido/lib/python3.12/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
INFO:AIDOCell:Loading gene annotations



Extracting: aido_cell_3m

1. Loading model and data...


/Users/sean/Desktop/GITHUB/napistu/lib/napistu-scrapyard/applications/foundation_models/.aido/lib/python3.12/site-packages/bionty/base/dev/_io.py:131: FutureWarning: Use synchronize_to instead of synchronize_to, synchronize_to will be removed in the future.
  remote_path.synchronize(localpath, error_no_origin=False, print_progress=True)


• standardized 19208/19264 terms


INFO:AIDOCell:Formatting model metadata


• standardized 19208/19264 terms


/Users/sean/Desktop/GITHUB/napistu/lib/napistu-scrapyard/applications/foundation_models/.aido/lib/python3.12/site-packages/bionty/base/dev/_io.py:131: FutureWarning: Use synchronize_to instead of synchronize_to, synchronize_to will be removed in the future.
  remote_path.synchronize(localpath, error_no_origin=False, print_progress=True)


   19264 genes, 6 layers
2. Extracting weights...
• standardized 19208/19264 terms


/Users/sean/Desktop/GITHUB/napistu/lib/napistu-scrapyard/applications/foundation_models/.aido/lib/python3.12/site-packages/bionty/base/dev/_io.py:131: FutureWarning: Use synchronize_to instead of synchronize_to, synchronize_to will be removed in the future.
  remote_path.synchronize(localpath, error_no_origin=False, print_progress=True)
INFO:utils:Saving weights to output/AIDOCell_weights.npz and metadata to output/AIDOCell_metadata.json
INFO:utils:Successfully validated weights, gene metadata and model metadata
INFO:utils:Saving weights to output/AIDOCell_weights.npz
INFO:utils:Saving metadata to output/AIDOCell_metadata.json
INFO:utils:Successfully saved all results


   Embeddings: (19264, 128)
   Attention weights: 6 layers × 4 matrices (Q,K,V,O)
3. Saving results...
   Successfully saved all results!


In [8]:
weights_dict, gene_annotations, model_metadata = load_results(OUTPUT_DIR, AIDOCELL_DEFS.MODEL_NAME)

GENES_OF_INTEREST = gene_annotations[RESULTS_DEFS.VOCAB_NAME].sample(10000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in model_metadata[RESULTS_DEFS.ORDERED_VOCABULARY]]

# Compute attention on demand
layer_attn = compute_attention_from_weights(
    weights_dict[RESULTS_DEFS.GENE_EMBEDDING][GENE_MASK,:],
    weights_dict[RESULTS_DEFS.ATTENTION_WEIGHTS]['layer_5'][RESULTS_DEFS.W_Q],
    weights_dict[RESULTS_DEFS.ATTENTION_WEIGHTS]['layer_5'][RESULTS_DEFS.W_K]
)

INFO:utils:Loading weights from output/AIDOCell_weights.npz and metadata from output/AIDOCell_metadata.json
INFO:utils:Loading weights from output/AIDOCell_weights.npz
INFO:utils:Loading metadata from output/AIDOCell_metadata.json
INFO:utils:Successfully loaded and validated all results
